# 05 · Shallow ConvNCF on MovieLens-1M

Thesis §5.6.1 · Table 14 · Figures 22–23

1. MF-BPR is pretrained for 15 epochs (Adagrad 0.05, 4 negatives per positive) to initialise the embeddings.
2. ConvNCF (outer-product map, Conv2D filters (32, 64, 64), kernel 2, max-pooling, dense 128) is warmed
   up for 1 epoch without regularisation.
3. The model is trained for 30 epochs with L2 = (emb 0.1, conv 0.1, out 1e-3), chosen from the grid
   {1e-3, 1e-1, 0}, and validated every 3 epochs. The best checkpoint is evaluated on the test split.

In [ ]:
# Folder containing ml-1m.ratings.dat and yelp.rating (see data/README.md).
# On Kaggle, point this at the attached dataset, e.g. "/kaggle/input/<dataset-name>".
DATA_DIR = "../data"

In [ ]:

import random, math
from functools import lru_cache
from collections import defaultdict
#HR (hit ratio)
def HR(ranked_predicted_list, true_test_item, number_of_rec_items):
    top_rec_items = ranked_predicted_list[:number_of_rec_items]
    item_ids = [item_id for item_id, _ in top_rec_items]
    if true_test_item in item_ids:
        return 1.0
    else:
        return 0.0
    
#Normalized Discounted Cumulative Gain (NDCG) 


def NDCG(ranked_predicted_list, true_test_item, number_of_rec_items):
    top_rec_items = ranked_predicted_list[:number_of_rec_items]
    item_ids = [item_id for item_id, _ in top_rec_items]
    for rank, item_id in enumerate(item_ids):
        if item_id == true_test_item:
            return 1.0/math.log2((rank+1) +1)
    return 0



#Get the predictions


#Evaluation using HR
def eval_hr_ndcg(
    interaction_matrix, interaction_array, test_df, model,
    number_of_rec_items, max_test_size, neg_sample_size
):
    # Build pivot mappings
    user_index_list = interaction_matrix.index.tolist()
    item_index_list = interaction_matrix.columns.tolist()

    user2pivot = {uid: idx for idx, uid in enumerate(user_index_list)}
    item2pivot = {iid: idx for idx, iid in enumerate(item_index_list)}

    # For each user (by pivot index), record the set of interacted item pivot indices
    user_item_dict = defaultdict(set)
    n_users, n_items = interaction_array.shape
    for u in range(n_users):
        user_item_dict[u] = set(np.where(interaction_array[u] == 1)[0])

    all_items = list(range(n_items))

    users = test_df['user_id'].values
    items = test_df['item_id'].values

    num_hits = 0.0
    num_test = 0.0
    sum_ndcg = 0.0

    for df_user, df_item in zip(users, items):
        if (df_user not in user2pivot) or (df_item not in item2pivot):
            continue

        user_idx = user2pivot[df_user]
        item_idx = item2pivot[df_item]

        items_interacted = user_item_dict[user_idx]
        possible_neg = list(set(all_items) - items_interacted)

        if len(possible_neg) == 0:
            # nothing to sample for this user; skip
            continue

        k = min(neg_sample_size, len(possible_neg))
        negative_sample = random.sample(possible_neg, k)

        test_sample_per_user = negative_sample + [item_idx]

        array_of_users = np.full(len(test_sample_per_user), user_idx, dtype='int32')
        array_of_items = np.array(test_sample_per_user, dtype='int32')

        predictions = model.predict([array_of_users, array_of_items], verbose=0).flatten()
        ranked_predicted_list = sorted(zip(test_sample_per_user, predictions),
                                       key=lambda x: x[1], reverse=True)

        hit = HR(ranked_predicted_list, item_idx, number_of_rec_items=number_of_rec_items)
        ndcg_val = NDCG(ranked_predicted_list, item_idx, number_of_rec_items)

        num_hits += hit
        sum_ndcg += ndcg_val
        num_test += 1

        

        if num_test >= max_test_size:
            break

    if num_test == 0:
        average_num_hits = 0.0
        ndcg_k = 0.0
    else:
        average_num_hits = num_hits / num_test
        ndcg_k = sum_ndcg / num_test

    
    return average_num_hits, ndcg_k


import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv(f"{DATA_DIR}/ml-1m.ratings.dat", sep="::", engine="python",
                 names=["user_id", "item_id", "rating", "timestamp"])
print("Original data:")
print(df.head())

#Check that every user has >=20 rankings

user_counts = df.groupby("user_id").size()
print("Minimum number of ratings per user:", user_counts.min())

# Sort the data by user, then by timestamp (ascending)
df = df.sort_values(['user_id', 'timestamp'])

# We will collect train rows and test rows in separate lists
train_list = []
test_list  = []

# Group by each user
for user_id, group in df.groupby('user_id'):
    # Convert group to list (or keep as DF if you like)
    user_history = group.values  # or group.to_numpy()

    # The last row of this user’s history is the "most recent" (test instance)
    test_interaction = user_history[-1]
    
    # All preceding rows are training data
    train_interactions = user_history[:-1]

    # Add them to our overall lists
    for row in train_interactions:
        train_list.append(row)
    test_list.append(test_interaction)

# Convert lists back to DataFrame
train_df = pd.DataFrame(train_list, columns=df.columns)
test_df  = pd.DataFrame(test_list,  columns=df.columns)
train_df = train_df.sort_values(['user_id', 'timestamp'])

val_df = train_df.groupby('user_id').tail(1).copy()   # 1 row per user
train_df_cv = train_df.drop(val_df.index).copy()

train_df = train_df_cv
print("Training size:", len(train_df))
print("Testing size: ", len(test_df))
# --- label-encode user/item to 0..n-1 (fit on TRAIN_CV + VAL + TEST) ---
ue, ie = LabelEncoder(), LabelEncoder()

all_ids = pd.concat([
    train_df[['user_id','item_id']],   # TRAIN_CV
    val_df[['user_id','item_id']],     # VAL  <-- include this
    test_df[['user_id','item_id']]     # TEST
], axis=0)

ue.fit(all_ids['user_id'])
ie.fit(all_ids['item_id'])

train_df['user_id'] = ue.transform(train_df['user_id'])
train_df['item_id'] = ie.transform(train_df['item_id'])
val_df['user_id']   = ue.transform(val_df['user_id'])    # <-- encode VAL
val_df['item_id']   = ie.transform(val_df['item_id'])    # <-- encode VAL
test_df['user_id']  = ue.transform(test_df['user_id'])
test_df['item_id']  = ie.transform(test_df['item_id'])

num_users = len(ue.classes_)
num_items = len(ie.classes_)

# --- per-user positive sets (from TRAIN_CV only, to avoid leakage) ---
user_pos = train_df.groupby('user_id')['item_id'].apply(set).to_dict()
all_items = np.arange(num_items, dtype=np.int32)

def make_bpr_triplets(user_pos, num_items, num_neg, seed=42):
    rng = np.random.default_rng(seed)
    users, pos_items, neg_items = [], [], []
    for u, pos_set in user_pos.items():
        pos_list = list(pos_set)
        for i in pos_list:
            for _ in range(num_neg):
                j = rng.integers(0, num_items)
                # resample until we hit a true negative
                while j in pos_set:
                    j = rng.integers(0, num_items)
                users.append(u); pos_items.append(i); neg_items.append(j)
    users = np.array(users, dtype=np.int32)
    pos_items = np.array(pos_items, dtype=np.int32)
    neg_items = np.array(neg_items, dtype=np.int32)
    return users, pos_items, neg_items


import tensorflow as tf
class BPRSequence(tf.keras.utils.Sequence):
    def __init__(self, user_pos, num_items, batch_size=1024, num_neg=1, seed=42):
        self.user_pos = user_pos
        self.num_items = num_items
        self.batch_size = batch_size
        self.num_neg = num_neg
        self.rng = np.random.default_rng(seed)
        self._rebuild()

    def _rebuild(self):
        self.u, self.i, self.j = make_bpr_triplets(
            self.user_pos,
            self.num_items,
            self.num_neg,
            seed=self.rng.integers(1_000_000)
            )
        self.n = self.u.shape[0]
        self.order = np.arange(self.n)
        self.rng.shuffle(self.order)

    def __len__(self):
        return int(np.ceil(self.n / self.batch_size))

    def __getitem__(self, idx):
        start = idx * self.batch_size
        end   = min(start + self.batch_size, self.n)
        bi    = self.order[start:end]

        X = {
            'user_id':     self.u[bi].astype(np.int32),
            'pos_item_id': self.i[bi].astype(np.int32),
            'neg_item_id': self.j[bi].astype(np.int32),
        }
        y = np.ones((len(bi), 1), dtype=np.float32)
        return X, y

    def on_epoch_end(self):
        self._rebuild()

# Build BPR sequence once and reuse everywhere (auto-resamples negatives each epoch)
train_seq = BPRSequence(user_pos, num_items, batch_size=256, num_neg=4)
import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers

def build_mf_scorer(num_users, num_items, emb_size):
    u_in = layers.Input(shape=(), dtype='int32', name='mf_user_id')
    i_in = layers.Input(shape=(), dtype='int32', name='mf_item_id')

    uE = layers.Embedding(num_users, emb_size, name='mf_user_emb')(u_in)   # no reg for pretrain
    iE = layers.Embedding(num_items, emb_size, name='mf_item_emb')(i_in)

    # dot product score: (B, K)·(B, K) -> (B, 1)
    score = layers.Dot(axes=-1, normalize=False, name='mf_dot')([uE, iE])
    return Model([u_in, i_in], score, name='MFScorer')

def build_mf_bpr_model(mf_scorer: tf.keras.Model):
    u_in  = layers.Input(shape=(), dtype='int32', name='user_id')
    i_pos = layers.Input(shape=(), dtype='int32', name='pos_item_id')
    i_neg = layers.Input(shape=(), dtype='int32', name='neg_item_id')

    s_pos = mf_scorer([u_in, i_pos])
    s_neg = mf_scorer([u_in, i_neg])
    diff  = layers.Subtract(name='logit_diff')([s_pos, s_neg])
    return Model([u_in, i_pos, i_neg], diff, name='MF_BPR')


def build_convncf_scorer(
        num_users,
        num_items,
        emb_size,
        conv_filters,
        kernel_size,
        dense_units,
        l2_emb=0.0,
        l2_conv=0.0,
        l2_out=0.0
    ):
    L2e = (lambda v: regularizers.l2(v) if v and v > 0 else None)

    user_input = layers.Input(shape=(), dtype='int32', name='sc_user_id')
    item_input = layers.Input(shape=(), dtype='int32', name='sc_item_id')

    user_emb = layers.Embedding(
        input_dim=num_users, output_dim=emb_size,
        embeddings_regularizer=L2e(l2_emb),
        name='user_embedding')(user_input)

    item_emb = layers.Embedding(
        input_dim=num_items, output_dim=emb_size,
        embeddings_regularizer=L2e(l2_emb),
        name='item_embedding')(item_input)

    u_exp = layers.Lambda(lambda x: tf.expand_dims(x, -1), name='u_expand')(user_emb)  # (B, emb, 1)
    v_exp = layers.Lambda(lambda x: tf.expand_dims(x,  1), name='v_expand')(item_emb)  # (B, 1, emb)
    E = layers.Multiply(name='outer_prod')([u_exp, v_exp])                              # (B, emb, emb)
    E = layers.Lambda(lambda x: tf.expand_dims(x, -1), name='add_channel')(E)          # (B, emb, emb, 1)

    x = E
    for idx, f in enumerate(conv_filters, 1):
        x = layers.Conv2D(
            filters=f, kernel_size=(kernel_size, kernel_size),
            activation='relu', padding='same',
            kernel_regularizer=L2e(l2_conv),
            name=f'conv{idx}')(x)
        x = layers.MaxPooling2D(pool_size=(2, 2), name=f'pool{idx}')(x)

    x = layers.Flatten(name='flatten')(x)
    x = layers.Dense(dense_units, activation='relu',
                     kernel_regularizer=L2e(l2_conv), name='dense')(x)

    score = layers.Dense(
        1, activation=None, name='score',
        kernel_regularizer=L2e(l2_out)
    )(x)

    return Model(inputs=[user_input, item_input], outputs=score, name='ConvNCFScorer')




def build_bpr_model(scorer: tf.keras.Model):
    u_in  = layers.Input(shape=(), dtype='int32', name='user_id')
    i_pos = layers.Input(shape=(), dtype='int32', name='pos_item_id')
    i_neg = layers.Input(shape=(), dtype='int32', name='neg_item_id')
    s_pos = scorer([u_in, i_pos])
    s_neg = scorer([u_in, i_neg])
    diff  = layers.Subtract(name='logit_diff')([s_pos, s_neg])
    return Model([u_in, i_pos, i_neg], diff, name='ConvNCF_BPR')


def pairwise_accuracy(y_true, logit_diff):
    return tf.reduce_mean(tf.cast(logit_diff > 0.0, tf.float32))
def pretrain_MF(BATCH_SIZE, NEG_PER_POS, MF_EPOCHS, emb_size=64):
    # 1) Build MF scorer instance
    mf_scorer = build_mf_scorer(num_users, num_items, emb_size=emb_size)
    mf_bpr    = build_mf_bpr_model(mf_scorer)

    mf_bpr.compile(
        optimizer=tf.keras.optimizers.Adagrad(0.05),
        loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
        metrics=[pairwise_accuracy]
    )

    # 2) Use a global train_seq (create it ONCE outside and reuse everywhere)
    global train_seq
    

    # 3) Train MF-BPR
    for ep in range(1, MF_EPOCHS + 1):
        mf_bpr.fit(train_seq, epochs=1, verbose=1)

    # 4) Grab pretrained embeddings
    P = mf_scorer.get_layer('mf_user_emb').get_weights()[0]
    Q = mf_scorer.get_layer('mf_item_emb').get_weights()[0]
    return P, Q

P, Q = pretrain_MF(BATCH_SIZE=256, NEG_PER_POS=4, MF_EPOCHS=15)



In [ ]:
# Make sure train_seq exists
try:
    train_seq
except NameError:
    train_seq = BPRSequence(user_pos, num_items, batch_size=256, num_neg=4)

rng = np.random.default_rng(2025)

# Build interaction matrix from TRAIN data (for leave-last-out eval)
val_inter_mat_sub = (
    train_df.assign(interaction=1)
            .pivot_table(index='user_id', columns='item_id',
                         values='interaction', fill_value=0)
)

# Align to full [0..num_users-1] × [0..num_items-1] space
val_inter_mat_sub = val_inter_mat_sub.reindex(
    index=np.arange(num_users),
    columns=np.arange(num_items),
    fill_value=0
)
val_inter_arr_sub = val_inter_mat_sub.values

# Validation set: use val_df as it is
val_df_sub = val_df.copy()

# 1) Build ConvNCF scorer without regularization and initialize from MF
scorer_no_reg = build_convncf_scorer(
    num_users=num_users,
    num_items=num_items,
    emb_size=64,
    conv_filters=(32, 64, 64),
    kernel_size=2,
    dense_units=128,
    l2_emb=0.0,
    l2_conv=0.0,
    l2_out=0.0
)

# init embeddings from MF pretrain
scorer_no_reg.get_layer('user_embedding').set_weights([P])
scorer_no_reg.get_layer('item_embedding').set_weights([Q])

bpr_no_reg = build_bpr_model(scorer_no_reg)

bpr_no_reg.compile(
    optimizer=tf.keras.optimizers.Adagrad(0.05),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[pairwise_accuracy]
)

# Warm-up: 1 epoch without regularization
bpr_no_reg.fit(train_seq, epochs=1, verbose=1)

# 2) Build ConvNCF scorer WITH regularization and transfer weights
scorer = build_convncf_scorer(
    num_users=num_users,
    num_items=num_items,
    emb_size=64,
    conv_filters=(32, 64, 64),
    kernel_size=2,
    dense_units=128,
    l2_emb=0.1,
    l2_conv=0.1,
    l2_out=0.001
)
scorer.set_weights(scorer_no_reg.get_weights())

bpr_model = build_bpr_model(scorer)
bpr_model.compile(
    optimizer=tf.keras.optimizers.Adagrad(0.05),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[pairwise_accuracy]
)

# 3) Train with regularization and monitor VAL HR/NDCG
hrs = []
ndcgs = []
weights = []
K = 10
VAL_NEG_SIZE = 99  # same as you used in eval_hr_ndcg for val/test
train_losses = []
train_pairwise_acc = []

for ep in range(1, 31):
    history = bpr_model.fit(train_seq, epochs=1, verbose=1)
    #append history 
    train_losses.append(history.history['loss'][0])
    train_pairwise_acc.append(history.history['pairwise_accuracy'][0])

    # every 3 epochs, evaluate on VAL set
    if ep % 3 == 0:
        hr, ndcg = eval_hr_ndcg(
            val_inter_mat_sub,
            val_inter_arr_sub,
            val_df_sub,
            scorer,                 # scorer used for prediction
            number_of_rec_items=K,
            max_test_size=10_000,
            neg_sample_size=VAL_NEG_SIZE
        )
        print(f"[VAL] epoch {ep:02d} — HR@{K}={hr:.4f}  NDCG@{K}={ndcg:.4f}")
        hrs.append(hr)
        ndcgs.append(ndcg)
        weights.append(scorer.get_weights())

# 4) Choose best epoch by NDCG and rebuild scorer with best weights
best_idx = int(np.argmax(ndcgs))
best_weights = weights[best_idx]

best_scorer = build_convncf_scorer(
    num_users=num_users,
    num_items=num_items,
    emb_size=64,
    conv_filters=(32, 64, 64),
    kernel_size=2,
    dense_units=128,
    l2_emb=0.1,
    l2_conv=0.1,
    l2_out=0.001
)
best_scorer.set_weights(best_weights)

# 5) Final TEST evaluation on test_df
for K_test in [5, 10, 20]:
    hr_t, ndcg_t = eval_hr_ndcg(
        val_inter_mat_sub,      # same interaction matrix (train interactions)
        val_inter_arr_sub,
        test_df,
        best_scorer,
        number_of_rec_items=K_test,
        max_test_size=10_000,
        neg_sample_size=VAL_NEG_SIZE
    )
    print(f"[TEST @ best setup] HR@{K_test}={hr_t:.4f}  NDCG@{K_test}={ndcg_t:.4f}")


print(hrs)
print(ndcgs)
